In [1]:
from sklearn import svm
from sklearn import svm
from tensorflow.keras.datasets import mnist
import numpy as np
from sklearn.decomposition import  KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import NearestCentroid
from sklearn.neighbors import KNeighborsClassifier

from sklearn import metrics
import time
import tensorflow as tf


Load dataset (MNIST or CIFAR-10)

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()            # Load MNIST dataset

#cifar10 = tf.keras.datasets.cifar10                                # Load CIFAR-10 dataset
#(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("Training data shape:", x_train.shape, y_train.shape)
print("Test data shape:", x_test.shape, y_test.shape)

y_train = np.squeeze(y_train)                                       # Convert y array to 1D - reduce dimensionality
y_test = np.squeeze(y_test)

__Preprocessing the data__

Perform regularization process

In [ ]:

x_train_flattened = x_train.reshape(x_train.shape[0], -1)           # Rearrange image data in 2 dimensions
x_test_flattened = x_test.reshape(x_test.shape[0], -1)

x_train_scaled = x_train_flattened.astype('float32') / 255.0        # Normalization for training and test sets
x_test_scaled = x_test_flattened.astype('float32') / 255.0


print("Training data scaler shape:", x_train_scaled.shape, y_train.shape)
print("Test data scaler shape:", x_test_scaled.shape, y_test.shape)

__Kernel Principal Component Analysis (PCA)__

The following code implements the KPCA dimensionality reduction process. 
The kernels tested are RBF and Polynomial with degrees 4 and 5. 
This particular technique uses a subset of the samples of the total training set as an attempt to use the whole set led to a memory error. 
The subset included tests on 30,000 and 40,000 samples.
n_components = components retained by KPCA, corresponds to information preservation during dimensionality reduction 
gamma = kernel coefficient - measures the effect of the kernel and the influence between samples when measuring their similarity 
small gamma value : greater generalization of the data - underfitting 
big gamma value : less influence of the samples - overfitting
the value gamma = None - automatically adjusts the parameter value based on the number of input features

In [ ]:
# Data subset
x_train = x_train_scaled[:40000]
y_train = y_train[:40000]

print(f"X_train shape: {x_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {x_test.shape}, y_test shape: {y_test.shape}") 

kpca = KernelPCA(n_components=300, kernel='poly', degree = 4, gamma=None)
print(f"n_components: {kpca.n_components}")

x_kpca_train = kpca.fit_transform(x_train)
x_kpca_test = kpca.transform(x_test_scaled)
print (x_kpca_train)
print(x_kpca_test.shape)


__Linear Discriminant Analysis (LDA)__

Code using LDA for dimensionality reduction  
The code reduces the number of features to 9 dimensions, keeping the most important information for separating the categories.  
n_componets : number of dimensions the data is transformed

In [ ]:
lda = LinearDiscriminantAnalysis(n_components = 9)                          # Create new feature dimensions
x_train_lda = lda.fit_transform(x_kpca_train, y_train)                      # Apply the LDA transformation
x_test_lda = lda.transform(x_kpca_test)
print(x_train_lda.shape)
print(x_test_lda.shape)


__Linear Kernel__

In [ ]:
svc = svm.SVC(probability = False, kernel = 'linear', C = 1)
    
svc.fit(x_train_lda, y_train)                                           # Training the model with the training data and the unique labels

svm_linear_train_model = svc.predict(x_train_lda)                       # Predict labels for each sample of x_train_reduced

acc_train = np.mean(svm_linear_train_model == y_train)                  # Calculating accuracy of correct predictions

print('Train Accuracy = {0:f}'.format(acc_train))    

Calculating predictions and accuracy on the test set using the trained learning model.

In [ ]:
svm_linear_test_model = svc.predict(x_test_lda)                          # Prediction process on new data

acc_test = np.mean(svm_linear_test_model == y_test)                      # Calculating forecast accuracy

print('Test Accuracy = {0:f}'.format(acc_test))

__Polynomial Kernel__

In [ ]:
svc_polynomial = svm.SVC(probability=False, kernel='poly', C=10, degree=2)       # Creating the SVC with polynomial kernel

svc_polynomial.fit(x_train_lda, y_train)                                         # SVC Training

svm_polynomial_train_model = svc_polynomial.predict(x_train_lda)                 # Applying the model to the training set

acc_train = np.mean(svm_polynomial_train_model == y_train)                       # Calculating accuracy

print('Accuracy = {0:f}'.format(acc_train))

Calculating predictions and accuracy on the test set using the trained learning model.

In [ ]:
svm_polynomial_test_model = svc_polynomial.predict(x_test_lda)                  # Predict labels for test_set

acc_test = np.mean(svm_polynomial_test_model == y_test)                         # Comparison of predicted and actual labels

print('Test Accuracy = {0:f}'.format(acc_test))


__Sigmoid Kernel__

In [ ]:
svm_sigmoid = svm.SVC(probability=False, kernel='sigmoid', C=0.1)               # Creating the SVC with sigmoid kernel

svm_sigmoid.fit(x_train_lda, y_train)                                           # SVC Training

svm_sigmoid_train_model = svm_sigmoid.predict(x_train_lda)                      # Calculate predictions on the training set

acc_train = np.mean(svm_sigmoid_train_model == y_train)                         # Calculating accuracy

print('Accuracy = {0:f}'.format(acc_train))


Calculating predictions and accuracy on the test set using the trained learning model.

In [ ]:
svm_sigmoid_test_model = svm_sigmoid.predict(x_test_lda)                        # Calculate predictions on the test set

acc_test = np.mean(svm_sigmoid_test_model == y_test)                            # Calculating the accuracy on the test set

print('Test Accuracy = {0:f}'.format(acc_test))                                 # Print the accuracy


__RBF Kernel__

In [ ]:
svc_rbf = svm.SVC(probability=False, kernel='rbf', C=10, gamma='scale')         # Creating the SVC with rdf kernel

svc_rbf.fit(x_train_lda, y_train)                                               # SVC Training

svm_rbf_train_model = svc_rbf.predict(x_train_lda)                              # Calculate predictions on the training set

acc_train = np.mean(svm_rbf_train_model == y_train)                             # Calculating accuracy

print('Accuracy = {0:f}'.format(acc_train))


Calculating predictions and accuracy on the test set using the trained learning model.

In [ ]:
svm_rbf_test_model = svc_rbf.predict(x_test_lda)                        # Calculate predictions on the test set

acc_test = np.mean(svm_rbf_test_model == y_test)                        # Calculating the accuracy on the test set

print('Test Accuracy = {0:f}'.format(acc_test))                         # Print the accuracy


__Nearest Neighbour Classification__

The Nearest Neighbor Classification algorithm is implemented to extract and compare results.  
The n_neighbors parameter is used to create the kNN model.  
This parameter determines the neighbors that should be considered for the classification of a new sample - based on how many nearest neighbor samples are used to place a new sample in a category.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors = 3)                                   # Create NN with k neighbors

knn_model.fit(x_train_lda, y_train)                                                 # Train the model with the data

print(knn_model)

print(x_test_lda.shape)

true_labels = y_test
predicted_labels = knn_model.predict(x_test_lda)

print(f"Classification Report:\n{metrics.classification_report(true_labels, predicted_labels)}\n")


__Nearest Centroid Classification__

Implementation of the Nearest Centroid Classification algorithm.  
The metric parameter determines how to calculate the distance between the new samples and the centers of the categories.  
The algorithm terminates in ms, which exceeds the ability of Notebook to time it.  
The time accuracy was calculated based on the time function

In [ ]:
start = int(round(time.time() * 1000))                                                  # Timing the process

centroid_classifier = NearestCentroid(metric = 'euclidean')                             # Model creation

centroid_classifier.fit(x_train_lda, y_train)

end = int(round(time.time() * 1000))

print("Finished in ", (end-start), "ms")
print(centroid_classifier)
print(x_test_lda.shape)

true_labels = y_test

predicted_labels = centroid_classifier.predict(x_test_lda)

print(f"Classification Report {centroid_classifier}:\n{metrics.classification_report(true_labels, predicted_labels)}\n")
